In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os
import numpy as np
import cv2
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [29]:
# ==============================
# 2. CONFIG
# ==============================
DATASET_PATH = "/content/drive/MyDrive/Dataset"   # change if needed
MODEL_PATH = "/content/drive/MyDrive/attention_unet_model.h5"

IMG_SIZE = (128, 128)
CROP = 10

In [30]:
# ==============================
# 3. LOAD MODEL (NO LOSS NEEDED)
# ==============================
model = load_model(MODEL_PATH, compile=False)

In [9]:
# ==============================
# 4. LOAD TEST DATA
# (handles your nested structure)
# ==============================
def load_test_images(base_dir, img_size=(128,128)):
    images = []
    labels = []

    for folder in os.listdir(base_dir):   # type1cam1, type2cam2...
        type_path = os.path.join(base_dir, folder)

        if not os.path.isdir(type_path):
            continue

        test_path = os.path.join(type_path, "test")
        if not os.path.exists(test_path):
            continue

        for label_name in ["good", "anomaly"]:
            class_path = os.path.join(test_path, label_name)

            if not os.path.exists(class_path):
                continue

            for img_file in os.listdir(class_path):
                img_path = os.path.join(class_path, img_file)

                img = cv2.imread(img_path)
                if img is None:
                    continue

                img = cv2.resize(img, img_size)
                img = img.astype(np.float32) / 255.0

                images.append(img)
                labels.append(0 if label_name == "good" else 1)

    return np.array(images), np.array(labels)


print("Loading test data...")
X_test, y_true = load_test_images(DATASET_PATH, IMG_SIZE)
print("Total test images:", len(X_test))


Loading test data...
Total test images: 1141


In [31]:
# ==============================
# 5. COMPUTE THRESHOLD
# (same logic as your training)
# ==============================
print("Computing threshold...")

val_errors = []

for img in X_test[:100]:   # small subset
    recon = model.predict(np.expand_dims(img, axis=0), verbose=0)[0]

    error_map = np.mean((img - recon) ** 2, axis=2)
    error_map = cv2.GaussianBlur(error_map, (5,5), 0)

    h, w = error_map.shape
    cropped = error_map[CROP:h-CROP, CROP:w-CROP]

    score = np.mean(np.sort(cropped.flatten())[-300:])
    val_errors.append(score)

threshold = np.mean(val_errors) + 2 * np.std(val_errors)

print("Threshold:", threshold)

Computing threshold...
Threshold: 0.002106667


In [32]:
# ==============================
# 6. PREDICTION (BATCH MODE)
# ==============================
print("Running inference...")

recons = model.predict(X_test, batch_size=16)

y_pred = []

for i in range(len(X_test)):
    img = X_test[i]
    recon = recons[i]

    error_map = np.mean((img - recon) ** 2, axis=2)
    error_map = cv2.GaussianBlur(error_map, (5,5), 0)

    h, w = error_map.shape
    cropped = error_map[CROP:h-CROP, CROP:w-CROP]

    score = np.mean(np.sort(cropped.flatten())[-300:])

    pred = 1 if score > threshold else 0
    y_pred.append(pred)

Running inference...
72/72 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step


In [33]:
# ==============================
# 7. METRICS
# ==============================
print("\n===== FINAL EVALUATION =====")

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1 Score :", f1_score(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))


===== FINAL EVALUATION =====
Accuracy : 0.5495179666958808
Precision: 0.9914529914529915
Recall   : 0.18441971383147854
F1 Score : 0.3109919571045576

Confusion Matrix:
[[511   1]
 [513 116]]
